In [ ]:
import dotenv
import os

dotenv.load_dotenv(dotenv_path='.env')

In [ ]:
import pyarrow as pa
import pandas
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError
from pyiceberg.transforms import IdentityTransform, DayTransform

In [ ]:
# 1. Connect to R2 Data Catalog

# Define catalog connection details
ENV = "prod"
CATALOG = "unprice-lakehouse-" + ENV
ACCOUNT_ID  = os.environ['ACCOUNT_ID']
CATALOG_URI = "https://catalog.cloudflarestorage.com/" + ACCOUNT_ID + "/" + CATALOG
TOKEN       = os.environ['TOKEN']
WAREHOUSE   = ACCOUNT_ID + "_" + CATALOG

# Connect to R2 Data Catalog
catalog = RestCatalog(
    name="unprice",
    warehouse=WAREHOUSE,
    uri=CATALOG_URI,
    token=TOKEN,
)

# list namespaces
print(catalog.list_namespaces())

In [ ]:
# drop tables
# catalog.drop_table('lakehouse.metadata')
# catalog.drop_table('lakehouse.entitlement_snapshot')
# catalog.drop_table('lakehouse.usage')
# catalog.drop_table('lakehouse.verification')

In [ ]:
# List tables in the "default" namespace
tables = catalog.list_tables(namespace='lakehouse')
print(tables)

In [ ]:
# Load and print schema for every table in the lakehouse namespace
tables = catalog.list_tables(namespace='lakehouse')
for table_id in tables:
    table = catalog.load_table(table_id)
    print(f"--- {table_id[0]}.{table_id[1]} ---")
    print(table.schema())
    print()

In [ ]:
table = catalog.load_table("lakehouse.events")
table.refresh()  # <-- this is the key
df = table.scan().to_arrow()
print(df)

In [ ]:
# List and peek contents of every table in the lakehouse namespace
import pandas as pd

MAX_ROWS = 1000  # rows to show per table

table = catalog.load_table("lakehouse.events")
scan = table.scan(limit=MAX_ROWS)
df = scan.to_pandas()
print(f"\n=== {name} (showing up to {MAX_ROWS} rows) ===")
if df.empty:
    print("(no rows)")
else:
    display(df)

In [ ]:
from pyiceberg.expressions import And, EqualTo

# Load your table
table = catalog.load_table("lakehouse.verification")  # adjust namespace.table_name
table.refresh()

seven_days_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()

# Build the expression tree
filter_expr = And(
    GreaterThanOrEqual("__ingest_ts", seven_days_ago),
    EqualTo("customer_id", "cus_11TqNF6bCebUjnx55pk6vs"),
    EqualTo("project_id", "proj_11STWG6AokEni2F3eQugHb")
)

# PyIceberg will use this to prune partitions/files and return only the matching rows
scan = table.scan(
    row_filter=filter_expr
)

# Get all matching files
files = []
for task in scan.plan_files():
    files.append({
        "file_path": task.file.file_path,
        "record_count": task.file.record_count,
        "file_size_in_bytes": task.file.file_size_in_bytes,
    })

print(f"Found {len(files)} files with {sum(f['record_count'] for f in files)} total records")
for f in files:
    print(f["file_path"])

In [ ]:
from pyiceberg.expressions import GreaterThanOrEqual, EqualTo, And
from datetime import datetime, timedelta

# Calculate 7 days ago dynamically
seven_days_ago = (datetime.utcnow() - timedelta(days=7)).isoformat()

# Build the expression tree
filter_expr = And(
    GreaterThanOrEqual("__ingest_ts", seven_days_ago),
    EqualTo("customer_id", "cus_11TqNF6bCebUjnx55pk6vs")
)

# PyIceberg will use this to prune partitions/files and return only the matching rows
scan = table.scan(
    row_filter=filter_expr
)

arrow_table = scan.to_arrow()

In [ ]:
# 1. Create namespace
# catalog.create_namespace("default")

In [ ]:
# 2. Load the pipeline-created table
# verifications = catalog.load_table("lakehouse.verification")
# print("Current Spec:", verifications.spec())
# metadata = catalog.load_table("lakehouse.metadata")
# print("Current Spec:", verifications.spec())
# entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
# print("Current Spec:", verifications.spec())
usage = catalog.load_table("default.usage3")
print("Current Spec:", usage.spec())
print(usage.metadata.location)

In [ ]:
old_partition_name = usage.spec().fields[0].name 
print(old_partition_name)

In [ ]:
# verifications
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.verification")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

In [ ]:
# usage
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("default.usage3")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # Remove the existing ingestion-time partition
    # update.remove_field(old_partition_name)

    # update.add_field("event_date", DayTransform(), old_partition_name)
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())


In [ ]:
# entitlement_snapshot
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.entitlement_snapshot")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

In [ ]:
# metadata
# Make sure to use the namespace defined in the pipeline (often 'default')
table = catalog.load_table("lakehouse.metadata")

# Add your new partition keys
# This example adds 'region' and 'account_id' as identity partitions
with table.update_spec() as update:
    # A. Remove all existing partition fields
    for field in table.spec().fields:
        # We remove by the partition field name (not source column name)
        update.remove_field(field.name)

    update.add_field("event_date", DayTransform(), "date_partition")
    update.add_field("project_id", IdentityTransform(), "project_partition")
    update.add_field("customer_id", IdentityTransform(), "customer_partition")

print("New Spec:", table.spec())
print("\nPartition Data Preview:")
print(table.inspect.partitions().to_pandas())

In [ ]:
# 4. Load the pipeline-created table
verifications = catalog.load_table("lakehouse.verification")
print("Current Spec:", verifications.spec())
metadata = catalog.load_table("lakehouse.metadata")
print("Current Spec:", verifications.spec())
entitlement_snapshot = catalog.load_table("lakehouse.entitlement_snapshot")
print("Current Spec:", verifications.spec())
usage = catalog.load_table("lakehouse.usage")
print("Current Spec:", verifications.spec())
print(verifications.metadata.location)